<a href="https://colab.research.google.com/github/sadhika-tech/deep-learning-lab/blob/main/Lab%204/vgg_sgd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train,y_train),(x_test,y_test)=cifar10.load_data()
class_names=['Airplane','Automobile','Bird','Cat','Deer','Dog','Frog','Horse','Ship','Truck']

x_train=x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print("Training labels after one-hot encoding:",y_train_cat.shape)

print("Testing labels after one-hot encoding:",y_test_cat.shape)

# Create a fresh VGG16 base
sgd_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)

sgd_base.trainable = False

x = sgd_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(10, activation="softmax")(x)

vgg_sgd = Model(
    inputs=sgd_base.input,
    outputs=output
)

vgg_sgd.compile(
    optimizer=SGD(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

sgd_history = vgg_sgd.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2119s 12us/step
Training labels after one-hot encoding: (50000, 10)
Testing labels after one-hot encoding: (10000, 10)
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.1675 - loss: 2.2697 - val_accuracy: 0.3298 - val_loss: 2.0590
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.2662 - loss: 2.0547 - val_accuracy: 0.3983 - val_loss: 1.9262
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.3135 - loss: 1.9448 - val_accuracy: 0.4233 - val_loss: 1.8322
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.3390 - loss: 1.8753 - val_accuracy: 0.4360 - val_loss: 1.7652
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.3635 - loss: 1.8180 - val_accuracy: 0.4447 - val_loss: 1.7137
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.3774 - loss: 1.7763 - val_accuracy: 0.4556 - val_loss: 1.6745
Ep

In [3]:
print("\nStarting Fine-Tuning...")


# Unfreeze all VGG16 layers
sgd_base.trainable = True

print("\nTrainable status of VGG16 layers:")

for layer in sgd_base.layers:
    print(layer.name, ":", layer.trainable)


# Recompile after unfreezing
vgg_sgd.compile(
    optimizer=SGD(
        learning_rate=0.0001,
        momentum=0.9
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# Fine-tune the complete model
fine_tune_history = vgg_sgd.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


Starting Fine-Tuning...

Trainable status of VGG16 layers:
input_layer : True
block1_conv1 : True
block1_conv2 : True
block1_pool : True
block2_conv1 : True
block2_conv2 : True
block2_pool : True
block3_conv1 : True
block3_conv2 : True
block3_conv3 : True
block3_pool : True
block4_conv1 : True
block4_conv2 : True
block4_conv3 : True
block4_pool : True
block5_conv1 : True
block5_conv2 : True
block5_conv3 : True
block5_pool : True
Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 54s 37ms/step - accuracy: 0.5998 - loss: 1.1414 - val_accuracy: 0.7026 - val_loss: 0.8424
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 45s 36ms/step - accuracy: 0.7091 - loss: 0.8431 - val_accuracy: 0.7492 - val_loss: 0.7059
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 45s 36ms/step - accuracy: 0.7555 - loss: 0.7170 - val_accuracy: 0.7664 - val_loss: 0.6646
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 45s 36ms/step - accuracy: 0.7817 - loss: 0.6390 - val_accuracy: 0.7807 - val_loss: 0.6198
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━

In [4]:
test_loss, test_accuracy = vgg_sgd.evaluate(
    x_test,
    y_test,
    verbose=1
)

print("\nFine-Tuned VGG16 Test Loss:", test_loss)
print("Fine-Tuned VGG16 Test Accuracy:", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8248 - loss: 0.5170

Fine-Tuned VGG16 Test Loss: 0.5169905424118042
Fine-Tuned VGG16 Test Accuracy: 0.8248000144958496


In [5]:
print("\nFrozen VGG16 Final Training Accuracy:",
      sgd_history.history["accuracy"][-1])

print("Frozen VGG16 Final Validation Accuracy:",
      sgd_history.history["val_accuracy"][-1])

print("Fine-Tuned VGG16 Final Training Accuracy:",
      fine_tune_history.history["accuracy"][-1])

print("Fine-Tuned VGG16 Final Validation Accuracy:",
      fine_tune_history.history["val_accuracy"][-1])



Frozen VGG16 Final Training Accuracy: 0.4192749857902527
Frozen VGG16 Final Validation Accuracy: 0.4742000102996826
Fine-Tuned VGG16 Final Training Accuracy: 0.8709750175476074
Fine-Tuned VGG16 Final Validation Accuracy: 0.8252000212669373
